# Baseline Models

Goal: set the performance floor that any real model must beat.
The test set is split off first and left untouched until the final evaluation.
All model comparison uses 5-fold stratified cross-validation on the training data only.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from sklearn.model_selection import train_test_split
from src.data import load_clean

df = load_clean("../data/telco_churn.csv")
X = df.drop(columns="Churn")
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.mean().round(4), y_test.mean().round(4))

(5634, 19) (1409, 19)
0.2654 0.2654


In [2]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
dummy = DummyClassifier(strategy="most_frequent")

res = cross_validate(dummy, X_train, y_train, cv=cv,
                     scoring=["accuracy", "roc_auc", "recall"])

for name in ["test_accuracy", "test_roc_auc", "test_recall"]:
    print(name, res[name].mean().round(3))

test_accuracy 0.735
test_roc_auc 0.5
test_recall 0.0


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()

preprocess = ColumnTransformer(
    [
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), cat_cols),
    ],
    remainder="passthrough",  # SeniorCitizen is already 0/1
)

models = {
    "logreg": LogisticRegression(max_iter=1000),
    "logreg_balanced": LogisticRegression(max_iter=1000, class_weight="balanced"),
}

rows = {}
for name, clf in models.items():
    pipe = Pipeline([("prep", preprocess), ("clf", clf)])
    res = cross_validate(
        pipe, X_train, y_train, cv=cv,
        scoring=["accuracy", "roc_auc", "recall", "precision"],
    )
    rows[name] = {m: res[f"test_{m}"].mean() for m in ["accuracy", "roc_auc", "recall", "precision"]}

pd.DataFrame(rows).T.round(3)

,accuracy,roc_auc,recall,precision
logreg,0.802,0.846,0.544,0.653
logreg_balanced,0.749,0.846,0.803,0.518


## Results: baselines (5-fold stratified CV on training data)

| Model | Accuracy | ROC-AUC | Recall | Precision |
|---|---|---|---|---|
| Always predict "no churn" | 0.735 | 0.500 | 0.000 | n/a |
| Logistic regression | 0.802 | 0.846 | 0.544 | 0.653 |
| Logistic regression (balanced) | 0.749 | 0.846 | 0.803 | 0.518 |

**Takeaways**
- The dummy baseline reaches 73.5% accuracy while catching no churners, which is why accuracy is not the headline metric here.
- Both logistic models reach ROC-AUC 0.846, so class weighting changes the operating point (the precision/recall trade-off), not the model's ranking ability.
- The preferred threshold should come from the relative cost of a retention offer versus a lost customer. I'll revisit this after trying stronger models.